# E2 — Quantum VQC + GAT-Transformer (Laptop)

**Architecture:** UNI(1024) → 128 → 3 qubits → measure(3) → concat(proj_3, q_3) → post_vqc(64) → +pos.enc(16) → 80 → GAT-Transformer

**Fixes vs the previous run that took ~100 min/epoch:**
- **Batched VQC** — `self.vqc(p)` processes all patches in one broadcasted call (no Python for-loop)
- **Better compression** — 1024 → 128 → 3 (not 1024 → 3 directly)
- **Data re-uploading** — AngleEmbedding at every layer (more expressive)
- **Post-VQC expansion** — 6 → 64 for fair comparison with classical baseline

**Expected:** < 15 min/epoch (down from ~100 min). 30 epochs → ~4 hours.

**Before running:** make sure `checkpoints/E2_quantum_*.pth` from previous runs are deleted — old `out_dim=6` checkpoints will not load into new `out_dim=64` model.

In [1]:
# Cell 1 — Setup
import os, sys, time, json, gc
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from sklearn.metrics import roc_auc_score, f1_score

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:512'

sys.path.insert(0, '..')
from pathq.model_v2   import QuantaPathV2
from pathq.dataset_v2 import get_loaders_from_features

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT     = Path('..').resolve()
FEAT_DIR = Path('data') / 'features_uni'   # relative to notebooks/ dir
CKPT_DIR = ROOT / 'checkpoints'
OUT_DIR  = ROOT / 'outputs'
CKPT_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

torch.manual_seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('CPU only')
print(f'FEAT : {FEAT_DIR.resolve()}')
print(f'CKPT : {CKPT_DIR}')

/home/kabi/.conda/envs/pathq/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[model_v2] Using Transformer for global branch
GPU  : NVIDIA GeForce RTX 5060 Laptop GPU
VRAM : 8.1 GB
FEAT : /home/kabi/PATHQ--Quantum-Digital-Pathology-for-Whole-Slide-Image-Analysis/notebooks/data/features_uni
CKPT : /home/kabi/PATHQ--Quantum-Digital-Pathology-for-Whole-Slide-Image-Analysis/checkpoints


In [2]:
# Cell 2 — Load data  (1024 patches — 3x faster than 3000)
# Switch to max_patches=3000 only after confirming fixes work
train_loader, val_loader, test_loader = get_loaders_from_features(
    features_dir = FEAT_DIR,
    batch_size   = 4,
    k            = 8,
    seed         = 42,
    max_patches  = 1024,
)
print(f'max_patches : 1024')
print(f'Train : {len(train_loader)} batches')
print(f'Val   : {len(val_loader)} batches')
print(f'Test  : {len(test_loader)} batches')

  Skipped 112 unlabeled file(s) (e.g. test_*) — keeping 221 labeled slides
Split: train=154 (pos=77) val=33 (pos=17) test=34 (pos=17)
max_patches : 1024
Train : 39 batches
Val   : 9 batches
Test  : 9 batches


In [3]:
# Cell 3 — Training functions
SEP  = '═' * 65
DASH = '─' * 65

def train_one(model, loader, optimizer, device):
    model.train()
    total, n = 0.0, 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        torch.cuda.empty_cache()
        logits, _ = model(batch)
        loss      = F.cross_entropy(logits, batch.y.view(-1), label_smoothing=0.1)
        loss_val  = loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        del logits, loss
        torch.cuda.empty_cache()
        total += loss_val; n += 1
    return total / max(n, 1)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    probs, labels, tl, n = [], [], 0.0, 0
    for batch in loader:
        batch     = batch.to(device)
        logits, _ = model(batch)
        tl       += F.cross_entropy(logits, batch.y.view(-1)).item()
        probs.extend(torch.softmax(logits, 1)[:, 1].cpu().tolist())
        labels.extend(batch.y.view(-1).cpu().tolist())
        n += 1
    p, l  = np.array(probs), np.array(labels)
    preds = (p >= 0.5).astype(int)
    auc   = roc_auc_score(l, p) if len(np.unique(l)) > 1 else 0.5
    f1    = f1_score(l, preds, zero_division=0)
    tp = int(((preds == 1) & (l == 1)).sum())
    fn = int(((preds == 0) & (l == 1)).sum())
    tn = int(((preds == 0) & (l == 0)).sum())
    fp = int(((preds == 1) & (l == 0)).sum())
    return {
        'auc'        : round(auc, 6),
        'f1'         : round(f1, 6),
        'loss'       : round(tl / max(n, 1), 6),
        'sensitivity': round(tp / max(tp + fn, 1), 4),
        'specificity': round(tn / max(tn + fp, 1), 4),
    }


def run_E2(model, tr, va, te, device,
           ckpt_best, ckpt_latest,
           epochs=40, lr=3e-5, patience=8):

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-2,
    )
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-7
    )
    plateau_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3, min_lr=1e-7
    )

    best_auc, pat, start = 0.0, 0, 1

    # Auto-resume — loads latest checkpoint if it exists and is compatible
    if Path(ckpt_latest).exists():
        try:
            ck = torch.load(ckpt_latest, weights_only=False)
            model.load_state_dict(ck['model_state'])
            start    = ck['epoch'] + 1
            best_auc = ck['best_auc']
            for _ in range(start - 1):
                cosine_sched.step()
            print(f'Resumed from epoch {start-1}  best_auc={best_auc:.4f}')
        except Exception as e:
            print(f'Checkpoint incompatible — starting fresh. ({e})')
            start, best_auc = 1, 0.0
    else:
        print('No checkpoint found — starting fresh.')

    print()
    print(SEP)
    print(' E2 Quantum VQC+GAT — max_patches=1024')
    print(' dropout=0.5 | weight_decay=1e-2 | label_smoothing=0.1')
    print(f' patience={patience} | epochs={epochs}')
    print(SEP)
    print(f' {"Ep":>3} {"TrL":>8} {"VaL":>8} {"VaAUC":>7} {"VaF1":>7} {"LR":>9} {"Secs":>6}')
    print(DASH)

    for ep in range(start, epochs + 1):
        t0   = time.time()
        tl   = train_one(model, tr, optimizer, device)
        vm   = evaluate(model, va, device)

        cosine_sched.step()
        plateau_sched.step(vm['auc'])

        current_lr = optimizer.param_groups[0]['lr']
        flag       = ''

        if vm['auc'] > best_auc:
            best_auc = vm['auc']
            pat      = 0
            flag     = '*'
            torch.save({'model_state': model.state_dict(),
                        'epoch': ep, 'best_auc': best_auc}, ckpt_best)
        else:
            pat += 1

        # Save every epoch — safe to interrupt and resume
        torch.save({'model_state': model.state_dict(),
                    'epoch': ep, 'best_auc': best_auc}, ckpt_latest)

        ow   = ' overfit' if vm['loss'] > tl * 2.5 else ''
        secs = int(time.time() - t0)
        print(f' {ep:>3} {tl:>8.4f} {vm["loss"]:>8.4f} '
              f'{vm["auc"]:>7.4f} {vm["f1"]:>7.4f} '
              f'{current_lr:>9.2e} {secs:>5}s {flag}{ow}')

        if pat >= patience:
            print()
            print(f' Early stop ep {ep} — patience={patience}')
            break

        torch.cuda.empty_cache()
        gc.collect()

    # Load best and evaluate on test set
    ck = torch.load(ckpt_best, weights_only=False)
    model.load_state_dict(ck['model_state'])
    tm = evaluate(model, te, device)

    print(DASH)
    print(f' Best val AUC : {best_auc:.4f}')
    print(f' Test AUC     : {tm["auc"]:.4f}')
    print(f' F1           : {tm["f1"]:.4f}')
    print(f' Sensitivity  : {tm["sensitivity"]:.4f}')
    print(f' Specificity  : {tm["specificity"]:.4f}')
    print(SEP)
    return {**tm, 'val_auc': best_auc,
            'gap': round(tm['auc'] - best_auc, 6)}

In [4]:
# Cell 4 — Run E2
CKPT_BEST   = str(CKPT_DIR / 'E2_quantum_best.pth')
CKPT_LATEST = str(CKPT_DIR / 'E2_quantum_latest.pth')

model_E2 = QuantaPathV2(
    use_vqc    = True,
    n_qubits   = 3,
    vqc_layers = 2,
    in_dim     = 1040,
).to(DEVICE)

n_p = sum(p.numel() for p in model_E2.parameters() if p.requires_grad)
print(f'Trainable params : {n_p:,}')
print(f'VQC out_dim      : {model_E2.vqc.out_dim}')
print()
print('Changes vs previous run:')
print('  max_patches  : 1024  (was 3000 — 3x faster epochs)')
print('  dropout      : 0.5   (was 0.4 — all layers)')
print('  weight_decay : 1e-2  (was 1e-3 — stronger L2)')
print('  label_smooth : 0.1   (new — prevents overconfident logits)')
print('  patience     : 8     (was 5)')
print('  schedulers   : cosine + plateau combined')
print('  epochs       : 40    (was 30)')
print()
print('Healthy training target:')
print('  Ep 5:  VaAUC ~0.70, VaLoss < 0.75')
print('  Ep 10: VaAUC ~0.75, VaLoss < 0.75')
print('  Ep 15: VaAUC ~0.80')
print()

result = run_E2(
    model_E2,
    train_loader, val_loader, test_loader,
    DEVICE,
    ckpt_best   = CKPT_BEST,
    ckpt_latest = CKPT_LATEST,
    epochs      = 40,
    lr          = 3e-5,
    patience    = 8,
)

with open(OUT_DIR / 'E2_result.json', 'w') as f:
    json.dump({
        'experiment'   : 'E2_quantum_restart',
        'max_patches'  : 1024,
        'dropout'      : 0.5,
        'weight_decay' : 1e-2,
        'label_smooth' : 0.1,
        'patience'     : 8,
        **result,
    }, f, indent=2)

print('E2 complete — run week6_xai.ipynb next')

[VQC] lightning.gpu  3q 2L — batched + re-uploading
QuantaPathV2: use_vqc=True, trainable=1,114,577
Trainable params : 1,114,577
VQC out_dim      : 64

Changes vs previous run:
  max_patches  : 1024  (was 3000 — 3x faster epochs)
  dropout      : 0.5   (was 0.4 — all layers)
  weight_decay : 1e-2  (was 1e-3 — stronger L2)
  label_smooth : 0.1   (new — prevents overconfident logits)
  patience     : 8     (was 5)
  schedulers   : cosine + plateau combined
  epochs       : 40    (was 30)

Healthy training target:
  Ep 5:  VaAUC ~0.70, VaLoss < 0.75
  Ep 10: VaAUC ~0.75, VaLoss < 0.75
  Ep 15: VaAUC ~0.80

No checkpoint found — starting fresh.

═════════════════════════════════════════════════════════════════
 E2 Quantum VQC+GAT — max_patches=1024
 dropout=0.5 | weight_decay=1e-2 | label_smoothing=0.1
 patience=8 | epochs=40
═════════════════════════════════════════════════════════════════
  Ep      TrL      VaL   VaAUC    VaF1        LR   Secs
────────────────────────────────────────────